# 16 — CTD DPO Boundary-Fixed Runner

DPO-only rerun for the CTD controlled benchmark. This notebook fixes the prompt/completion token-boundary mismatch seen in Experiment 15. It reuses the saved Robust SFT adapters from `results/15/adapters/` and does **not** rerun Mixture SFT or confidence abstention.

Key fix: DPO completions begin with an explicit leading space, preserving the exact raw prompt format used by the Robust SFT model while preventing BPE merges across `Answer:` and the first completion token. A fail-fast preflight verifies the token prefix for both chosen and rejected responses before any DPO training starts.


In [ ]:
!pip -q install -U transformers datasets trl peft accelerate bitsandbytes sentencepiece requests

import os,re,gc,json,random,gzip
from pathlib import Path
import numpy as np, pandas as pd, torch, requests
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import PeftModel, prepare_model_for_kbit_training
from trl import DPOConfig, DPOTrainer

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
SEEDS = [1,2,3]
SPLITS = ['ChemicalID','GeneID','DiseaseID']
N_TRAIN = 1500
N_EVAL = 100
DPO_STEPS = 60
DPO_BETA = 0.1
DPO_LR = 5e-6

ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
DATA_DIR = ROOT/'ctd_data'; DATA_DIR.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/llm-tuning-playground')
except Exception:
    DRIVE_ROOT = ROOT/'llm-tuning-playground'

ROBUST_DIR = DRIVE_ROOT/'results/15/adapters'
RESULT_DIR = DRIVE_ROOT/'results/16'
ADAPTER_DIR = RESULT_DIR/'adapters'
RESULT_DIR.mkdir(parents=True, exist_ok=True); ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
RESULT_CSV = RESULT_DIR/'16_dpo_fixed_results.csv'
SUMMARY_CSV = RESULT_DIR/'16_dpo_fixed_summary.csv'
CONFIG_JSON = RESULT_DIR/'16_dpo_fixed_config.json'

config = dict(model=MODEL_NAME,seeds=SEEDS,splits=SPLITS,n_train=N_TRAIN,n_eval=N_EVAL,dpo_steps=DPO_STEPS,beta=DPO_BETA,learning_rate=DPO_LR,source_robust_adapters=str(ROBUST_DIR),fix='leading-space completion + token-prefix preflight')
CONFIG_JSON.write_text(json.dumps(config,indent=2), encoding='utf-8')
print('CUDA:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Results:', RESULT_DIR)


In [ ]:
# CTD acquisition + robust parser (same construction as Experiment 14/15)
CHEM_NAME='CTD_chem_gene_ixns.tsv.gz'
GD_NAMES=['CTD_curated_genes_diseases.tsv.gz','CTD_genes_diseases.tsv.gz']

def valid_gzip(path,min_bytes=10000):
    path=Path(path)
    if not path.exists() or path.stat().st_size<min_bytes:return False
    try:
        with open(path,'rb') as f:
            if f.read(2)!=b'\x1f\x8b':return False
        with gzip.open(path,'rb') as f:f.read(256)
        return True
    except Exception:return False

def find_local(name):
    for p in [Path.cwd()/name,ROOT/name,DATA_DIR/name,Path('/content/drive/MyDrive')/name,Path('/content/drive/MyDrive/ctd')/name,Path('/content/drive/MyDrive/data')/name]:
        if valid_gzip(p): print('Found:',p); return p
    return None

def download_ctd(name):
    dest=DATA_DIR/name
    for url in [f'https://ctdbase.org/reports/{name}',f'https://ctdbase.org/downloads/{name}',f'http://ctdbase.org/reports/{name}']:
        try:
            print('Trying:',url)
            with requests.get(url,stream=True,timeout=(20,300),allow_redirects=True,headers={'User-Agent':'Mozilla/5.0'}) as r:
                r.raise_for_status()
                with open(dest,'wb') as f:
                    for ch in r.iter_content(1024*1024):
                        if ch:f.write(ch)
            if valid_gzip(dest): print('Downloaded:',dest); return dest
        except Exception as e: print(' failed:',type(e).__name__,str(e)[:120])
        dest.unlink(missing_ok=True)
    return None

def ensure_ctd(names):
    if isinstance(names,str): names=[names]
    for n in names:
        p=find_local(n)
        if p:return p
    for n in names:
        p=download_ctd(n)
        if p:return p
    raise FileNotFoundError('Could not obtain CTD data: '+', '.join(names))

def read_ctd(path,expected_any):
    header=None
    with gzip.open(path,'rt',encoding='utf-8',errors='replace') as f:
        for line in f:
            if not line.startswith('#'):break
            s=line.lstrip('#').strip()
            if '\t' in s:
                cols=[x.strip() for x in s.split('\t')]
                if any(x in cols for x in expected_any): header=cols
    if header is None: raise ValueError(f'Could not recover CTD header from {path}')
    return pd.read_csv(path,sep='\t',comment='#',compression='gzip',dtype=str,low_memory=False,header=None,names=header)

def pick(df,names):
    for n in names:
        if n in df.columns:return n
    raise KeyError(f'None of {names} found')

CHEM_GENE=ensure_ctd(CHEM_NAME); GENE_DISEASE=ensure_ctd(GD_NAMES)
cg=read_ctd(CHEM_GENE,['ChemicalName','ChemicalID','GeneSymbol','GeneID'])
gd=read_ctd(GENE_DISEASE,['GeneSymbol','GeneID','DiseaseName','DiseaseID'])
c_name=pick(cg,['ChemicalName']); c_id=pick(cg,['ChemicalID']); g_sym1=pick(cg,['GeneSymbol']); g_id1=pick(cg,['GeneID'])
g_sym2=pick(gd,['GeneSymbol']); g_id2=pick(gd,['GeneID']); d_name=pick(gd,['DiseaseName']); d_id=pick(gd,['DiseaseID'])
cg2=cg[[c_name,c_id,g_sym1,g_id1]].dropna().drop_duplicates(); gd2=gd[[g_sym2,g_id2,d_name,d_id]].dropna().drop_duplicates()
cg2.columns=['ChemicalName','ChemicalID','GeneSymbol','GeneID']; gd2.columns=['GeneSymbol','GeneID','DiseaseName','DiseaseID']
paths=cg2.merge(gd2,on=['GeneSymbol','GeneID'],how='inner').drop_duplicates()
paths=paths[(paths.ChemicalName.str.len()<100)&(paths.DiseaseName.str.len()<120)].reset_index(drop=True)
edge_pool=gd2[['GeneSymbol','DiseaseName']].drop_duplicates().reset_index(drop=True)
assert len(paths)>3000
print('Two-hop paths:',len(paths))


In [ ]:
# Benchmark construction + evaluation helpers
def render_prompt(row,edges):
    lines=[f'- {g} -> {d}' for g,d in edges]
    return ('Use only the supplied evidence. Determine the disease supported by the path from the queried chemical through the queried gene. '
            'If no supplied gene-disease relation supports the queried gene, answer exactly: No supported path.\n\n'
            f'Chemical: {row.ChemicalName}\nGene: {row.GeneSymbol}\nEvidence:\n'+'\n'.join(lines))

def positive_edges(row,k,rng):
    edges=[(str(row.GeneSymbol),str(row.DiseaseName))]
    pool=edge_pool[(edge_pool.GeneSymbol!=row.GeneSymbol)&(edge_pool.DiseaseName!=row.DiseaseName)]
    if k:
        sub=pool.sample(n=k,random_state=rng.randint(0,2**31-1)); edges += [(str(g),str(d)) for g,d in sub.itertuples(index=False,name=None)]
    rng.shuffle(edges); return edges

def no_path_edges(row,k,rng,lexical=False):
    pool=edge_pool[(edge_pool.GeneSymbol!=row.GeneSymbol)&(edge_pool.DiseaseName!=row.DiseaseName)].copy(); edges=[]
    if lexical:
        sym=str(row.GeneSymbol); near=pool[pool.GeneSymbol.astype(str).str.startswith(sym[:max(1,min(2,len(sym)))])]
        if len(near):
            x=near.sample(1,random_state=rng.randint(0,2**31-1)).iloc[0]; edges.append((str(x.GeneSymbol),str(x.DiseaseName))); pool=pool[pool.GeneSymbol!=x.GeneSymbol]
    need=k-len(edges)
    if need>0:
        sub=pool.sample(need,random_state=rng.randint(0,2**31-1)); edges += [(str(g),str(d)) for g,d in sub.itertuples(index=False,name=None)]
    rng.shuffle(edges); return edges

def counterfactual_edges(row,rng):
    c=edge_pool[(edge_pool.GeneSymbol==row.GeneSymbol)&(edge_pool.DiseaseName!=row.DiseaseName)]
    cf=str(c.sample(1,random_state=rng.randint(0,2**31-1)).iloc[0].DiseaseName) if len(c) else str(edge_pool[edge_pool.DiseaseName!=row.DiseaseName].sample(1,random_state=rng.randint(0,2**31-1)).iloc[0].DiseaseName)
    return [(str(row.GeneSymbol),cf)],cf

def answer_text(row): return f'Disease: {row.DiseaseName}. Reasoning: {row.ChemicalName} -> {row.GeneSymbol} -> {row.DiseaseName}.'

def make_split(df,col,seed):
    r=np.random.default_rng(seed); ents=df[col].dropna().unique().copy(); r.shuffle(ents); cut=max(1,int(.8*len(ents)))
    tr_e,te_e=set(ents[:cut]),set(ents[cut:]); trp=df[df[col].isin(tr_e)]; tep=df[df[col].isin(te_e)].drop_duplicates(['ChemicalID','GeneID','DiseaseID'])
    assert len(trp)>=N_TRAIN and len(tep)>=N_EVAL
    tr=trp.sample(N_TRAIN,random_state=seed).reset_index(drop=True); te=tep.sample(N_EVAL,random_state=1000+seed).reset_index(drop=True)
    assert set(tr[col]).isdisjoint(set(te[col])); return tr,te

def item(row,edges,target,typ): return {'target_gene':str(row.GeneSymbol),'target_disease':None if target is None else str(target),'evidence_edges':[(str(g),str(d)) for g,d in edges],'prompt':render_prompt(row,edges),'answer_type':typ}

def make_eval_sets(df,seed):
    rng=random.Random(20000+seed); out={k:[] for k in ['clean','distractor_5','hard_no_path','lexical_no_path','counterfactual']}
    for _,row in df.iterrows():
        out['clean'].append(item(row,positive_edges(row,0,rng),row.DiseaseName,'positive'))
        out['distractor_5'].append(item(row,positive_edges(row,5,rng),row.DiseaseName,'positive'))
        out['hard_no_path'].append(item(row,no_path_edges(row,5,rng,False),None,'no_path'))
        out['lexical_no_path'].append(item(row,no_path_edges(row,5,rng,True),None,'no_path'))
        e,cf=counterfactual_edges(row,rng); out['counterfactual'].append(item(row,e,cf,'positive'))
    return out

def norm(s): return re.sub(r'\s+',' ',str(s).strip().lower())
def score_one(x,p):
    p=norm(p)
    if x['answer_type']=='no_path': return 'no supported path' in p
    return norm(x['target_disease']) in p and 'no supported path' not in p
def score_set(items,preds): return float(np.mean([score_one(x,p) for x,p in zip(items,preds)]))


In [ ]:
# DPO pair construction — fixed boundary
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME,use_fast=True)
tokenizer.pad_token=tokenizer.pad_token or tokenizer.eos_token
tokenizer.padding_side='left'

def make_dpo_dataset(df,seed):
    rng=random.Random(30000+seed); rec=[]
    for _,row in df.iterrows():
        if rng.random() < 0.5:
            edges=no_path_edges(row,rng.choice([3,5,10]),rng,rng.random()<0.5)
            rejected_disease=edges[0][1]
            chosen='No supported path.'
            rejected=f'Disease: {rejected_disease}.'
        else:
            edges=positive_edges(row,rng.choice([1,3,5,10]),rng)
            distractors=[d for g,d in edges if g != str(row.GeneSymbol)]
            rejected_disease=distractors[0] if distractors else 'Unknown disease'
            chosen=answer_text(row)
            rejected=f'Disease: {rejected_disease}.'
        prompt=render_prompt(row,edges)+'\nAnswer:'
        # Critical fix: force a whitespace boundary so Qwen BPE cannot merge `Answer:` with the first answer token.
        rec.append({'prompt':prompt,'chosen':' '+chosen,'rejected':' '+rejected})
    return Dataset.from_list(rec)

def boundary_preflight(ds,n=100):
    n=min(n,len(ds)); bad=[]
    for i in range(n):
        ex=ds[i]; p=tokenizer(ex['prompt'],add_special_tokens=False).input_ids
        for key in ('chosen','rejected'):
            full=tokenizer(ex['prompt']+ex[key],add_special_tokens=False).input_ids
            if full[:len(p)] != p:
                bad.append((i,key,len(p),full[:len(p)]==p))
    if bad:
        raise RuntimeError(f'DPO token-boundary preflight FAILED: {bad[:10]}')
    print(f'DPO token-boundary preflight: {n}/{n} examples aligned for chosen + rejected ✅')

# Quick global sanity check before allocating a training model.
_tr,_=make_split(paths,'ChemicalID',SEEDS[0])
_preview=make_dpo_dataset(_tr,SEEDS[0])
boundary_preflight(_preview,100)
print('Example prompt suffix:',repr(_preview[0]['prompt'][-30:]))
print('Chosen prefix:',repr(_preview[0]['chosen'][:30]))
del _tr,_preview


In [ ]:
# Model loading, DPO training, and deterministic evaluation
compute_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=compute_dtype,bnb_4bit_use_double_quant=True)

def load_robust_adapter(adapter_path,training=False):
    base=AutoModelForCausalLM.from_pretrained(MODEL_NAME,quantization_config=bnb,device_map='auto')
    base.config.use_cache=not training
    if training: base=prepare_model_for_kbit_training(base)
    m=PeftModel.from_pretrained(base,str(adapter_path),is_trainable=training)
    return m

def train_dpo(dpo_ds,robust_adapter,outdir,seed):
    set_seed(seed); boundary_preflight(dpo_ds,100)
    m=load_robust_adapter(robust_adapter,training=True)
    kwargs=dict(output_dir=str(outdir),per_device_train_batch_size=2,gradient_accumulation_steps=8,max_steps=DPO_STEPS,learning_rate=DPO_LR,beta=DPO_BETA,logging_steps=20,save_strategy='no',report_to='none',remove_unused_columns=False,gradient_checkpointing=True,max_length=512,max_prompt_length=384)
    if compute_dtype==torch.bfloat16: kwargs['bf16']=True
    else: kwargs['fp16']=True
    # Version-adaptive DPOConfig for recent TRL releases.
    try:
        args=DPOConfig(**kwargs)
    except TypeError:
        kwargs.pop('max_prompt_length',None)
        try: args=DPOConfig(**kwargs)
        except TypeError:
            kwargs.pop('max_length',None); args=DPOConfig(**kwargs)
    try:
        trainer=DPOTrainer(model=m,ref_model=None,args=args,train_dataset=dpo_ds,processing_class=tokenizer)
    except TypeError:
        trainer=DPOTrainer(model=m,ref_model=None,args=args,train_dataset=dpo_ds,tokenizer=tokenizer)
    trainer.train()
    trainer.model.save_pretrained(outdir)
    return trainer.model

@torch.no_grad()
def predict_items(model,items,batch_size=8):
    model.eval(); preds=[]
    old_side=tokenizer.padding_side; tokenizer.padding_side='left'
    for i in range(0,len(items),batch_size):
        batch=items[i:i+batch_size]
        prompts=[x['prompt']+'\nAnswer:' for x in batch]
        enc=tokenizer(prompts,return_tensors='pt',padding=True,truncation=True,max_length=512).to(model.device)
        out=model.generate(**enc,max_new_tokens=64,do_sample=False,pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
        plen=enc['input_ids'].shape[1]
        preds += tokenizer.batch_decode(out[:,plen:],skip_special_tokens=True)
    tokenizer.padding_side=old_side
    return preds

def cleanup_model(model):
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


In [ ]:
# DPO-only runner with per-(split, seed) checkpointing
if RESULT_CSV.exists():
    rows=pd.read_csv(RESULT_CSV).to_dict('records')
else:
    rows=[]

def complete(split,seed):
    got={(r['condition']) for r in rows if r['split']==split and int(r['seed'])==seed}
    return set(['clean','distractor_5','hard_no_path','lexical_no_path','counterfactual']).issubset(got)

def checkpoint():
    df=pd.DataFrame(rows); df.to_csv(RESULT_CSV,index=False)
    if len(df):
        summary=df.groupby(['split','condition'],as_index=False)['accuracy'].agg(['mean','std']).reset_index()
        summary.to_csv(SUMMARY_CSV,index=False)
    print('Checkpointed rows:',len(rows))

for split in SPLITS:
    for seed in SEEDS:
        print('\n'+'='*80); print('SPLIT',split,'SEED',seed); print('='*80)
        if complete(split,seed):
            print('Already complete — skipping.'); continue
        robust_adapter=ROBUST_DIR/f'robust_{split}_{seed}'
        if not robust_adapter.exists():
            raise FileNotFoundError(f'Missing Robust adapter: {robust_adapter}. Run/restore the Experiment 15 Robust reference first; this notebook intentionally does not retrain other methods.')
        train_df,test_df=make_split(paths,split,seed)
        dpo_ds=make_dpo_dataset(train_df,seed)
        boundary_preflight(dpo_ds,100)
        outdir=ADAPTER_DIR/f'dpo_fixed_{split}_{seed}'; outdir.mkdir(parents=True,exist_ok=True)
        print('Training boundary-fixed DPO from:',robust_adapter)
        model=train_dpo(dpo_ds,robust_adapter,outdir,seed)
        eval_sets=make_eval_sets(test_df,seed)
        for condition,items in eval_sets.items():
            preds=predict_items(model,items)
            acc=score_set(items,preds)
            print(condition,acc)
            rows=[r for r in rows if not (r['split']==split and int(r['seed'])==seed and r['condition']==condition)]
            rows.append({'split':split,'seed':seed,'method':'dpo_fixed','condition':condition,'accuracy':acc})
            checkpoint()
        cleanup_model(model)

checkpoint()
print('DONE')
print('Results:',RESULT_CSV)
print('Summary:',SUMMARY_CSV)
